# KKBOX Tab 1 Pre-Expiry Pulse Context

Notebook nay sinh artifact pulse theo ngay cho thang truoc `target_month`.

Vi du:

- `TARGET_MONTH = 201704`
- pulse context duoc dung de trinh bay la `201703`
- cohort van la cohort sap het han cua `201704`

Output chinh:

- `artifacts_tab1_preexpiry_pulse/tab1_preexpiry_pulse_daily_201704.parquet`
- `artifacts_tab1_preexpiry_pulse/tab1_preexpiry_pulse_summary_201704.json`
- `artifacts_tab1_preexpiry_pulse/manifest.json`


In [ ]:
from __future__ import annotations

from datetime import date, datetime, timezone
from pathlib import Path
import gc
import json
import shutil
import subprocess
from typing import Iterable, Iterator, Optional

import numpy as np
import pandas as pd

FEATURE_STORE_ROOT_HINT = None
RAW_DATA_ROOT_HINT = None
OUTPUT_DIR = "artifacts_tab1_preexpiry_pulse"
TARGET_MONTH = 201704

LOW_ACTIVITY_THRESHOLD_SECS = 1800.0
HIGH_RISK_THRESHOLD = 0.60
CHUNK_SIZE_TX = 2_000_000
CHUNK_SIZE_LOG = 2_000_000

SEVEN_ZIP_BIN = shutil.which("7z") or shutil.which("7za") or shutil.which("7zr")
KAGGLE_COMPETITION_DIR = Path("/kaggle/input/competitions/kkbox-churn-prediction-challenge")

RAW_TX_FILES = ["transactions_v2.csv", "transactions.csv"]
RAW_LOG_FILES = ["user_logs_v2.csv", "user_logs.csv"]


In [ ]:
def month_start_from_yyyymm(yyyymm: int) -> date:
    text = str(int(yyyymm)).zfill(6)
    return date(int(text[:4]), int(text[4:6]), 1)


def previous_month(yyyymm: int) -> int:
    value = month_start_from_yyyymm(yyyymm)
    if value.month == 1:
        return (value.year - 1) * 100 + 12
    return value.year * 100 + (value.month - 1)


def yyyymm_label(yyyymm: int) -> str:
    text = str(int(yyyymm)).zfill(6)
    return text[:4] + "-" + text[4:6]


def resolve_feature_store_dir(root_hint: Optional[str | Path] = None, score_month: int = 201704) -> Path:
    required = [
        "train_features_bi_all.parquet",
        f"test_features_bi_{score_month}_full.parquet",
        "feature_columns.csv",
        "bi_dimension_columns.csv",
    ]
    candidates: list[Path] = []
    if root_hint is not None:
        root_path = Path(root_hint)
        candidates.extend([root_path, root_path / "feature_store", root_path / "data" / "artifacts" / "feature_store"])
    candidates.extend(
        [
            Path("/kaggle/input/kkbox-feature-store"),
            Path("/kaggle/input/kkbox-feature-store/feature_store"),
            Path("/kaggle/input/kkbox-churn-feature-store"),
            Path("/kaggle/input/kkbox-churn-feature-store/feature_store"),
            Path("/kaggle/input/kkbox-churn-output"),
            Path("/kaggle/input/kkbox-churn-output/feature_store"),
        ]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate.exists() and all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError("Cannot resolve feature store dataset for Kaggle notebook.")


def resolve_raw_data_dir(root_hint: Optional[str | Path] = None) -> Path:
    candidates: list[Path] = []
    if root_hint is not None:
        root_path = Path(root_hint)
        candidates.extend([root_path, root_path / "raw"])
    candidates.extend(
        [
            KAGGLE_COMPETITION_DIR,
            Path("/kaggle/input/kkbox-churn-prediction-challenge"),
            Path("/kaggle/input/kkbox-raw"),
            Path("/kaggle/input/kkbox-raw-data"),
        ]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Cannot resolve raw KKBOX dataset for Kaggle notebook.")


def resolve_source_file(file_name: str, data_dir: Path, required: bool = True) -> Optional[Path]:
    csv_path = data_dir / file_name
    archive_path = data_dir / f"{file_name}.7z"
    if csv_path.exists():
        return csv_path
    if archive_path.exists():
        return archive_path
    if required:
        raise FileNotFoundError(f"Missing source file: {file_name} or {file_name}.7z in {data_dir}")
    return None


def iter_csv_chunks(
    data_dir: Path,
    file_name: str,
    *,
    usecols: Iterable[str],
    dtype: dict[str, str],
    chunksize: int,
) -> Iterator[pd.DataFrame]:
    source_path = resolve_source_file(file_name, data_dir)
    assert source_path is not None

    if source_path.suffix == ".csv":
        yield from pd.read_csv(source_path, usecols=list(usecols), dtype=dtype, chunksize=chunksize)
        return

    if SEVEN_ZIP_BIN is None:
        raise RuntimeError("7z binary is required to stream .7z files on Kaggle.")

    process = subprocess.Popen(
        [SEVEN_ZIP_BIN, "x", "-so", str(source_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    try:
        assert process.stdout is not None
        for chunk in pd.read_csv(process.stdout, usecols=list(usecols), dtype=dtype, chunksize=chunksize):
            yield chunk
        process.stdout.close()
        stderr_text = process.stderr.read().decode("utf-8", errors="ignore") if process.stderr is not None else ""
        return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(f"7z failed for {source_path}: {stderr_text}")
    except Exception:
        process.kill()
        process.wait()
        raise


def load_feature_snapshot(feature_store_dir: Path, score_month: int, columns: Iterable[str]) -> pd.DataFrame:
    score_path = feature_store_dir / f"test_features_bi_{score_month}_full.parquet"
    if score_path.exists():
        df = pd.read_parquet(score_path, columns=list(columns))
        if "target_month" in df.columns:
            df = df[df["target_month"] == score_month].reset_index(drop=True)
        return df
    master_path = feature_store_dir / "bi_feature_master.parquet"
    if master_path.exists():
        df = pd.read_parquet(master_path, columns=list(columns))
        return df[df["target_month"] == score_month].reset_index(drop=True)
    raise FileNotFoundError(f"Missing feature snapshot for score_month={score_month}")


In [ ]:
TARGET_MONTH_LABEL = yyyymm_label(TARGET_MONTH)
CONTEXT_MONTH = previous_month(TARGET_MONTH)
CONTEXT_MONTH_LABEL = yyyymm_label(CONTEXT_MONTH)
CONTEXT_START = month_start_from_yyyymm(CONTEXT_MONTH)
TARGET_START = month_start_from_yyyymm(TARGET_MONTH)
CONTEXT_START_INT = int(CONTEXT_START.strftime("%Y%m%d"))
TARGET_START_INT = int(TARGET_START.strftime("%Y%m%d"))

feature_store_dir = resolve_feature_store_dir(FEATURE_STORE_ROOT_HINT, score_month=TARGET_MONTH)
raw_data_dir = resolve_raw_data_dir(RAW_DATA_ROOT_HINT)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

cohort_columns = [
    "msno",
    "target_month",
    "skip_ratio",
    "discovery_ratio",
    "is_auto_renew",
    "is_manual_renew",
    "high_skip_flag",
    "low_discovery_flag",
    "content_fatigue_flag",
]
cohort_df = load_feature_snapshot(feature_store_dir, TARGET_MONTH, cohort_columns).copy()
cohort_df["msno"] = cohort_df["msno"].astype(str)
cohort_df = cohort_df.drop_duplicates(subset=["msno"]).reset_index(drop=True)

for col in ["skip_ratio", "discovery_ratio", "is_auto_renew", "is_manual_renew", "high_skip_flag", "low_discovery_flag", "content_fatigue_flag"]:
    cohort_df[col] = pd.to_numeric(cohort_df[col], errors="coerce").fillna(0.0).astype("float32")

manual_renew_signal = np.where(cohort_df["is_manual_renew"] > 0, cohort_df["is_manual_renew"], 1.0 - cohort_df["is_auto_renew"])
cohort_df["manual_renew_signal"] = np.clip(manual_renew_signal, 0.0, 1.0).astype("float32")
cohort_df["base_risk"] = (
    0.25 * np.clip(cohort_df["skip_ratio"], 0.0, 1.0)
    + 0.15 * (1.0 - np.clip(cohort_df["discovery_ratio"], 0.0, 1.0))
    + 0.15 * cohort_df["manual_renew_signal"]
    + 0.10 * np.clip(cohort_df["high_skip_flag"], 0.0, 1.0)
    + 0.05 * np.clip(cohort_df["low_discovery_flag"], 0.0, 1.0)
    + 0.05 * np.clip(cohort_df["content_fatigue_flag"], 0.0, 1.0)
).clip(0.0, 1.0).astype("float32")
cohort_df["no_log_risk"] = (cohort_df["base_risk"] + 0.18).clip(0.0, 1.0).astype("float32")

cohort_profile = cohort_df[["msno", "base_risk", "no_log_risk"]].copy()
cohort_msnos = set(cohort_profile["msno"].tolist())
cohort_size = int(len(cohort_profile))
base_no_log_risk_sum = float(cohort_profile["no_log_risk"].sum())
base_no_log_high_risk_count = int((cohort_profile["no_log_risk"] >= HIGH_RISK_THRESHOLD).sum())

print(json.dumps({
    "target_month": TARGET_MONTH_LABEL,
    "context_month": CONTEXT_MONTH_LABEL,
    "cohort_size": cohort_size,
    "feature_store_dir": str(feature_store_dir),
    "raw_data_dir": str(raw_data_dir),
}, ensure_ascii=False, indent=2))


In [ ]:
tx_usecols = ["msno", "transaction_date", "actual_amount_paid"]
tx_dtype = {
    "transaction_date": "int32",
    "actual_amount_paid": "float32",
}

tx_parts: list[pd.DataFrame] = []
for file_name in RAW_TX_FILES:
    if resolve_source_file(file_name, raw_data_dir, required=False) is None:
        continue
    print(f"Reading transaction source: {file_name}")
    for chunk in iter_csv_chunks(raw_data_dir, file_name, usecols=tx_usecols, dtype=tx_dtype, chunksize=CHUNK_SIZE_TX):
        chunk["msno"] = chunk["msno"].astype(str)
        subset = chunk.loc[
            chunk["msno"].isin(cohort_msnos)
            & chunk["transaction_date"].between(CONTEXT_START_INT, TARGET_START_INT - 1)
        ].copy()
        if subset.empty:
            continue
        subset["event_date"] = pd.to_datetime(subset["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
        subset = subset.dropna(subset=["event_date"])
        grouped = subset.groupby("event_date", as_index=False).agg(
            total_revenue=("actual_amount_paid", "sum"),
            total_transactions=("msno", "size"),
        )
        tx_parts.append(grouped)
        del subset, grouped, chunk
        gc.collect()

if tx_parts:
    tx_daily = pd.concat(tx_parts, ignore_index=True)
    tx_daily = tx_daily.groupby("event_date", as_index=False).sum(numeric_only=True)
else:
    tx_daily = pd.DataFrame(columns=["event_date", "total_revenue", "total_transactions"])

tx_daily["total_revenue"] = pd.to_numeric(tx_daily.get("total_revenue", 0.0), errors="coerce").fillna(0.0)
tx_daily["total_transactions"] = pd.to_numeric(tx_daily.get("total_transactions", 0), errors="coerce").fillna(0).astype(int)
print(tx_daily.head())


In [ ]:
log_usecols = ["msno", "date", "num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
log_dtype = {
    "date": "int32",
    "num_25": "float32",
    "num_50": "float32",
    "num_75": "float32",
    "num_985": "float32",
    "num_100": "float32",
    "num_unq": "float32",
    "total_secs": "float32",
}

log_parts: list[pd.DataFrame] = []
for file_name in RAW_LOG_FILES:
    if resolve_source_file(file_name, raw_data_dir, required=False) is None:
        continue
    print(f"Reading user-log source: {file_name}")
    for chunk in iter_csv_chunks(raw_data_dir, file_name, usecols=log_usecols, dtype=log_dtype, chunksize=CHUNK_SIZE_LOG):
        chunk["msno"] = chunk["msno"].astype(str)
        subset = chunk.loc[
            chunk["msno"].isin(cohort_msnos)
            & chunk["date"].between(CONTEXT_START_INT, TARGET_START_INT - 1)
        ].copy()
        if subset.empty:
            continue
        subset["event_date"] = pd.to_datetime(subset["date"].astype(str), format="%Y%m%d", errors="coerce")
        subset = subset.dropna(subset=["event_date"])
        subset["skip_num"] = subset["num_25"] + subset["num_50"]
        subset["play_num"] = subset[["num_25", "num_50", "num_75", "num_985", "num_100"]].sum(axis=1)
        subset["disc_num"] = subset["num_unq"]
        grouped = subset.groupby(["msno", "event_date"], as_index=False).agg(
            total_secs=("total_secs", "sum"),
            skip_num=("skip_num", "sum"),
            play_num=("play_num", "sum"),
            disc_num=("disc_num", "sum"),
        )
        log_parts.append(grouped)
        del subset, grouped, chunk
        gc.collect()

if log_parts:
    log_user_day = pd.concat(log_parts, ignore_index=True)
    log_user_day = log_user_day.groupby(["msno", "event_date"], as_index=False).sum(numeric_only=True)
else:
    log_user_day = pd.DataFrame(columns=["msno", "event_date", "total_secs", "skip_num", "play_num", "disc_num"])

log_user_day = log_user_day.merge(cohort_profile, on="msno", how="left")
for col in ["total_secs", "skip_num", "play_num", "disc_num", "base_risk", "no_log_risk"]:
    log_user_day[col] = pd.to_numeric(log_user_day[col], errors="coerce").fillna(0.0)

log_user_day["day_skip_ratio"] = np.where(log_user_day["play_num"] > 0, log_user_day["skip_num"] / log_user_day["play_num"], 0.0)
log_user_day["day_discovery_ratio"] = np.where(log_user_day["play_num"] > 0, log_user_day["disc_num"] / log_user_day["play_num"], 0.0)
log_user_day["low_activity_flag"] = (log_user_day["total_secs"] < LOW_ACTIVITY_THRESHOLD_SECS).astype("float32")
log_user_day["risk_score"] = (
    log_user_day["base_risk"]
    + 0.15 * log_user_day["low_activity_flag"]
    + 0.07 * np.clip(log_user_day["day_skip_ratio"], 0.0, 1.0)
    + 0.03 * (1.0 - np.clip(log_user_day["day_discovery_ratio"], 0.0, 1.0))
).clip(0.0, 1.0)
log_user_day["high_risk_user"] = (log_user_day["risk_score"] >= HIGH_RISK_THRESHOLD).astype(int)
log_user_day["baseline_high_risk_user"] = (log_user_day["no_log_risk"] >= HIGH_RISK_THRESHOLD).astype(int)

if not log_user_day.empty:
    risk_daily = log_user_day.groupby("event_date", as_index=False).agg(
        active_users=("msno", "nunique"),
        total_listening_secs=("total_secs", "sum"),
        active_risk_sum=("risk_score", "sum"),
        active_no_log_risk_sum=("no_log_risk", "sum"),
        active_high_risk_users=("high_risk_user", "sum"),
        active_baseline_high_risk_users=("baseline_high_risk_user", "sum"),
    )
    risk_daily["total_risk_sum"] = base_no_log_risk_sum + (risk_daily["active_risk_sum"] - risk_daily["active_no_log_risk_sum"])
    risk_daily["high_risk_users"] = base_no_log_high_risk_count + (
        risk_daily["active_high_risk_users"] - risk_daily["active_baseline_high_risk_users"]
    )
    risk_daily["avg_risk_score"] = np.where(cohort_size > 0, (risk_daily["total_risk_sum"] / cohort_size) * 100.0, 0.0)
    risk_daily["high_risk_users"] = risk_daily["high_risk_users"].round().astype(int)
    risk_daily["active_users"] = risk_daily["active_users"].round().astype(int)
else:
    risk_daily = pd.DataFrame(columns=["event_date", "active_users", "total_listening_secs", "high_risk_users", "avg_risk_score"])

risk_daily = risk_daily[["event_date", "active_users", "total_listening_secs", "high_risk_users", "avg_risk_score"]]
print(risk_daily.head())


In [ ]:
date_frame = pd.DataFrame({
    "event_date": pd.date_range(CONTEXT_START, TARGET_START - pd.Timedelta(days=1), freq="D")
})

pulse_daily = date_frame.merge(tx_daily, on="event_date", how="left")
pulse_daily = pulse_daily.merge(risk_daily, on="event_date", how="left")

for col in ["total_revenue", "total_transactions", "active_users", "total_listening_secs", "high_risk_users", "avg_risk_score"]:
    pulse_daily[col] = pd.to_numeric(pulse_daily[col], errors="coerce").fillna(0.0)

pulse_daily["total_transactions"] = pulse_daily["total_transactions"].round().astype(int)
pulse_daily["active_users"] = pulse_daily["active_users"].round().astype(int)
pulse_daily["high_risk_users"] = pulse_daily["high_risk_users"].round().astype(int)
pulse_daily["event_date"] = pulse_daily["event_date"].dt.date.astype(str)
pulse_daily["target_month"] = TARGET_MONTH
pulse_daily["target_month_label"] = TARGET_MONTH_LABEL
pulse_daily["context_month"] = CONTEXT_MONTH
pulse_daily["context_month_label"] = CONTEXT_MONTH_LABEL
pulse_daily["cohort_size"] = cohort_size
pulse_daily["series_mode"] = "pre_expiry_context"

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_month": TARGET_MONTH,
    "target_month_label": TARGET_MONTH_LABEL,
    "context_month": CONTEXT_MONTH,
    "context_month_label": CONTEXT_MONTH_LABEL,
    "cohort_size": cohort_size,
    "low_activity_threshold_secs": LOW_ACTIVITY_THRESHOLD_SECS,
    "high_risk_threshold": HIGH_RISK_THRESHOLD,
    "feature_store_dir": str(feature_store_dir),
    "raw_data_dir": str(raw_data_dir),
    "output_files": {
        "daily": f"tab1_preexpiry_pulse_daily_{TARGET_MONTH}.parquet",
        "summary": f"tab1_preexpiry_pulse_summary_{TARGET_MONTH}.json",
        "manifest": "manifest.json",
    },
}
manifest_payload = {
    "notebook_name": "kkbox-preexpiry-pulse-context.ipynb",
    "artifact_type": "tab1_preexpiry_pulse",
    "created_at_utc": summary_payload["generated_at_utc"],
    "inputs": {
        "feature_store_dir": str(feature_store_dir),
        "raw_data_dir": str(raw_data_dir),
    },
    "outputs": {
        "daily": str(output_dir / f"tab1_preexpiry_pulse_daily_{TARGET_MONTH}.parquet"),
        "summary": str(output_dir / f"tab1_preexpiry_pulse_summary_{TARGET_MONTH}.json"),
    },
    "metadata": {
        "target_month": TARGET_MONTH,
        "context_month": CONTEXT_MONTH,
        "cohort_size": cohort_size,
        "series_mode": "pre_expiry_context",
    },
}

daily_path = output_dir / f"tab1_preexpiry_pulse_daily_{TARGET_MONTH}.parquet"
summary_path = output_dir / f"tab1_preexpiry_pulse_summary_{TARGET_MONTH}.json"
manifest_path = output_dir / "manifest.json"

pulse_daily.to_parquet(daily_path, index=False)
summary_path.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding="utf-8")
manifest_path.write_text(json.dumps(manifest_payload, ensure_ascii=False, indent=2), encoding="utf-8")

display(pulse_daily.head())
print(json.dumps(summary_payload, ensure_ascii=False, indent=2))


## Output contract cho API sau nay

Parquet daily co cac cot can cho pulse panel:

- `event_date`
- `total_revenue`
- `total_transactions`
- `high_risk_users`
- `avg_risk_score`
- `active_users`
- `total_listening_secs`
- `target_month`, `target_month_label`
- `context_month`, `context_month_label`
- `cohort_size`
- `series_mode`

Khi keo len dashboard:

- `target_month = 201704`
- `context_month = 201703`
- frontend co the noi ro day la pulse context cua thang truoc de support du bao / simulation cho `2017-04`.
